In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as job_manager
import lib_dna_member.coupon_digital_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_coupon_and_digital(job):
    """
    Generate the features associated with coupon and digital

    Parameters:
        job (managers.JobManager): object which manages the Spark App
    Returns:
        (pyspark.sql.DataFrame): dna with new features
    """
    dna = job.tables["population"]
    
    orig_cols = dna.columns

    weeks_back = [4, 8, 12, 26, 52]

    dna = features.feature_atc(job, dna, weeks_back)
    
    dna = features.feature_coupon_clipped(job, dna, weeks_back)
    
    # dna = utils.cache_df(dna)

    tender_type_cds = ["CPN", "PCUM", "PCUS", "PCUE", "PCUB", "PCUR"]

    # Calculate the total number of coupon redemptions within
    # a given fiscal week
    dna = features.feature_fiscal_coupon_general(
        job,
        dna,
        tender_type_cds=tender_type_cds,
        discount_type_cds=["ZCOU", "ZPAP"],
        aggregate=f.count("*"),
        alias="FW_COUPON_REDEMPTIONS",
    )
    

    # Calculate the total number of coupon redemptions + clipless within
    # a given fiscal week
    dna = features.feature_fiscal_coupon_general(
        job,
        dna,
        tender_type_cds=tender_type_cds,
        discount_type_cds=["ZCOU", "ZPAP", "ZCLP"],
        aggregate=f.count("*"),
        alias="FW_COUPON_REDEMPTIONS_W_CLPLSS",
    )
    
    # Calculate the total fiscal amount redeemed in coupons within
    # a given fiscal week
    dna = features.feature_fiscal_coupon_general(
        job,
        dna,
        tender_type_cds=tender_type_cds,
        discount_type_cds=["ZCOU", "ZPAP"],
        aggregate=f.sum("savings_temp"),
        alias="FW_COUPON_SAVINGS",
    )
    
    # Calculate the total fiscal amount redeemed in coupons + in clipless
    # within a given fiscal week
    dna = features.feature_fiscal_coupon_general(
        job,
        dna,
        tender_type_cds=tender_type_cds,
        discount_type_cds=["ZCOU", "ZPAP", "ZCLP"],
        aggregate=f.sum("savings_temp"),
        alias="FW_COUPON_SAVINGS_W_CLPLSS",
    )
    
    dna = features.feature_days_since_last_coupon_redeemed(job, dna)
    
    dna = features.feature_days_since_last_email_open(job, dna)
    
    dna = features.feature_days_since_last_atc_clipped(job, dna)
    
    dna = features.feature_email_open_rate(job, dna)
    
    dna = features.feature_fiscal_savings_with_clipless(job, dna)
    
    dna = features.feature_cpn_channel(job, dna)
    
    # dna = utils.cache_df(dna)

    # Compute the aggregate fiscal amount redeemed in coupons + clipless
    # per customer {weeks} fiscal weeks back
    weeks = ["FOUR", "EIGHT", "TWELVE", "TWENTY-SIX", "FIFTY-TWO"]
    for num_weeks in weeks:
        dna = features.feature_aggregate_per_member_weeks(
            job, dna, num_weeks, "FW_SAVINGS_W_CLPLSS"
        )
        

    # dna = utils.cache_df(dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    silver_skeleton, silver_master_member_extended, silver_master_member_history, silver_transaction_fiscal_header, silver_transaction_fiscal_detail, silver_coupon_clip_fiscal, silver_master_email, 
    silver_master_email_fiscal, silver_awards_fiscal, silver_transaction_fiscal_payment,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)


In [0]:
job.read_table("skeleton") 
job.read_table("member_extended")
job.read_table("member_history")
job.read_table("header_fiscal")
job.read_table("detail_fiscal")
job.read_table("payment_fiscal")
job.read_table("coupon_clip_fiscal")
job.read_table("email")
job.read_table("email_fiscal")
job.read_table("awards_fiscal")

In [0]:
job.tables["header_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["header_fiscal"]
)

job.tables["detail_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["detail_fiscal"]
)

job.tables["payment_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["payment_fiscal"]
)

job.tables["coupon_clip_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["coupon_clip_fiscal"]
)

job.tables["awards_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["awards_fiscal"]
)

population = gp.generate_population(job)
job.tables["population"] = population

features = generate_coupon_and_digital(job)
features = features.withColumn('MBRSHP_SID', f.coalesce('MBRSHP_SID',f.lit(-1)))

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_coupon_and_digital_1}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_coupon_and_digital_1,
    df=features,
    mode="merge"
)